In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
50,19,Male,101.0,Mild,Delhi,Yes
39,50,Female,103.0,Mild,Kolkata,No
95,12,Female,104.0,Mild,Bangalore,No
68,54,Female,104.0,Strong,Kolkata,No
96,51,Female,101.0,Strong,Kolkata,Yes


In [7]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [8]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [10]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),
                                                df['has_covid'],
                                                test_size= 0.2)

In [12]:
x_train

,age,gender,fever,cough,city
21,73,Male,98.0,Mild,Bangalore
65,69,Female,102.0,Mild,Bangalore
43,22,Female,99.0,Mild,Bangalore
53,83,Male,98.0,Mild,Delhi
37,55,Male,100.0,Mild,Kolkata
...,...,...,...,...,...
61,81,Female,98.0,Strong,Mumbai
68,54,Female,104.0,Strong,Kolkata
96,51,Female,101.0,Strong,Kolkata
0,60,Male,103.0,Mild,Kolkata


### 1. Aam Zindagi (long route to do column transformation) 

In [16]:
# Adding simple Imputer to fever column
si =SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])

# Also the Test data
x_test_fever = si.fit_transform(x_test[['fever']])

x_train_fever.shape

(80, 1)

In [17]:
# Ordinal Encoding -> cough
oe = OrdinalEncoder(categories=[["Mild","Strong"]])
x_train_cough = oe.fit_transform(x_train[['cough']])

# Also the test data
x_test_cough = oe.fit_transform(x_test[['cough']])

x_train_cough.shape

(80, 1)

In [18]:
# One Hot Encoding -> gender , city
ohe = OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

# Also the test data
x_test_gender_city = ohe.fit_transform(x_test[['gender','city']])

x_train_gender_city.shape

(80, 4)

In [19]:
# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

# Also from test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city'])

x_train_age.shape

(80, 1)

In [20]:
# Now we will concatinate everything
x_train_transformed = np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)

# same step for test data
x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

x_train_transformed.shape

(80, 7)

### Mentos Zindagi (Simpler Way)

In [21]:
from sklearn.compose import ColumnTransformer

In [23]:
ct = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [25]:
ct.fit_transform(x_train).shape

(80, 7)